In [1]:
import os, glob
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import osmnx as ox
import geopandas as gpd

# Get Espoo boundary as a GeoDataFrame (admin boundary relation)
espoo = ox.geocode_to_gdf("Espoo, Finland")

# Often returns a multipolygon boundary geometry
espoo_boundary = espoo[["display_name", "geometry"]].copy()
espoo_boundary

,display_name,geometry
0,"Espoo, Helsinki sub-region, Uusimaa, Mainland ...","POLYGON ((24.49929 60.28759, 24.50779 60.28612..."


In [3]:
espoo_boundary.explore()

In [9]:
fin_pop_raster_files = {
    0: r'data\population\fin_t_00_2025_CN_100m_R2025A_v1.tif',
    1: r'data\population\fin_t_01_2025_CN_100m_R2025A_v1.tif',
    5: r'data\population\fin_t_05_2025_CN_100m_R2025A_v1.tif',
    10: r'data\population\fin_t_10_2025_CN_100m_R2025A_v1.tif',
    15: r'data\population\fin_t_15_2025_CN_100m_R2025A_v1.tif',
}

In [10]:
young_pop_espoo = []

for pop, raster in fin_pop_raster_files.items():
    source_fin = rasterio.open(raster)

    geom_fin = espoo_boundary.geometry.values
    
    clipped_fin, clipped_transform = mask(
                                            source_fin,
                                            geom_fin,
                                            crop=True,
                                            nodata=source_fin.nodata
                                        )

    clipped_array = clipped_fin[0] 
    mask_arr = clipped_array != source_fin.nodata
    
    records = []
    for geom, value in shapes(
                                clipped_array,
                                mask=mask_arr,
                                transform=clipped_transform
                            ):
                                records.append({
                                    "geometry": shape(geom),
                                    f"pop_{pop}": value
                                })
    pop_espoo = gpd.GeoDataFrame(records, crs=source_fin.crs)

    young_pop_espoo.append(pop_espoo)

In [12]:
len(young_pop_espoo)

young_pop_espoo

[                                                geometry     pop_0
 0      POLYGON ((24.665 60.36333, 24.665 60.3625, 24....  0.000702
 1      POLYGON ((24.65 60.36083, 24.65 60.36, 24.6508...  0.002151
 2      POLYGON ((24.65083 60.36083, 24.65083 60.36, 2...  0.001255
 3      POLYGON ((24.65167 60.36083, 24.65167 60.36, 2...  0.001000
 4      POLYGON ((24.6525 60.36083, 24.6525 60.36, 24....  0.001408
 ...                                                  ...       ...
 46888  POLYGON ((24.75833 60.06083, 24.75833 60.06, 2...  0.003293
 46889  POLYGON ((24.75917 60.06083, 24.75917 60.06, 2...  0.001200
 46890  POLYGON ((24.75667 60.06, 24.75667 60.05917, 2...  0.001091
 46891  POLYGON ((24.7575 60.06, 24.7575 60.05917, 24....  0.001038
 46892  POLYGON ((24.75833 60.06, 24.75833 60.05917, 2...  0.000837
 
 [46893 rows x 2 columns],
                                                 geometry     pop_1
 0      POLYGON ((24.665 60.36333, 24.665 60.3625, 24....  0.002996
 1      POLYGON ((2

In [30]:
import pandas as pd
import geopandas as gpd

gdfs_indexed = [gdf.set_index("geometry") for gdf in young_pop_espoo]

merged = pd.concat(gdfs_indexed, axis=1).fillna(0)
merged_gdf = gpd.GeoDataFrame(merged.reset_index(), geometry="geometry")

merged_gdf["population_sum"] = (
    merged_gdf.drop(columns="geometry").sum(axis=1)
)

In [31]:
merged_gdf.head()

,geometry,pop_0,pop_1,pop_5,pop_10,pop_15,population_sum
0,"POLYGON ((24.665 60.36333, 24.665 60.3625, 24....",0.000702,0.002996,0.003958,0.004473,0.004420,0.016549
1,"POLYGON ((24.65 60.36083, 24.65 60.36, 24.6508...",0.002151,0.009184,0.012135,0.013712,0.013550,0.050732
2,"POLYGON ((24.65083 60.36083, 24.65083 60.36, 2...",0.001255,0.005356,0.007077,0.007997,0.007902,0.029587
3,"POLYGON ((24.65167 60.36083, 24.65167 60.36, 2...",0.001000,0.004271,0.005643,0.006377,0.006301,0.023592
4,"POLYGON ((24.6525 60.36083, 24.6525 60.36, 24....",0.001408,0.006010,0.007941,0.008973,0.008867,0.033199


In [33]:
merged_gdf.to_file("output/espoo_children_population.geojson", driver="GeoJSON")